# Gender Coding Companies

### Inputs: 
- `"../data/derived/industry_gender_share.csv"` 
- `"../data/derived/company_profiles.csv"` 

### Outputs:
- `"../data/derived/company_gender_share.csv"`
- `"../data/derived/company_gender_share.xlsx"`

  

### Purpose:

Map each company to an industry category with a known percentage of women workers. 

Some companies remain unmatched, which need to be supplemented by manual review.

In [ ]:
import pandas as pd
from langchain_dartmouth.llms import ChatDartmouthCloud
from langchain_core.prompts import ChatPromptTemplate

from joblib import Parallel, delayed
from tqdm.auto import tqdm

In [ ]:
industry_gender = pd.read_csv("../data/derived/industry_gender_share.csv")
industry_gender.head()

In [ ]:
industries = industry_gender.industry.to_list()

In [ ]:
company_profiles = pd.read_csv("../data/derived/company_profiles.csv")

In [ ]:
company_profiles.head()

In [ ]:
company_profiles = company_profiles[
    [
        "id",
        "company_name",
        "about",
        "specialties",
        "organization_type",
        "industries",
        "unformatted_about",
    ]
]

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

In [ ]:
def get_census_industry(company_profile: pd.Series) -> dict:
    llm = ChatDartmouthCloud(
        model_name="google_genai.gemini-2.0-flash-001",
        max_tokens=1024,
    )
    output_schema = """```json
{
  "type": "object",
  "properties": {
      "id": {
      "type": "string",
      "description": "Unique identifier for the organization"
    },
      "name": {
      "type": "string",
      "description": "Name of the organization"
    },
      "assessment": {
      "type": "string",
      "description": "Detailed assessment or description of the organization"
    },
      "industry": {
      "type": "string",
      "description": "The industry classification"
    }
  },
  "required": ["id", "name", "assessment", "industry"],
  "additionalProperties": false
}
```
"""

    industry_prompt = ChatPromptTemplate(
        [
            (
                "system",
                "Your task is to identify the best matching "
                "industry category from a set of options for a given company. Discuss the data "
                "given in the company profile before responding with your "
                "final decision with a valid JSON object using the following schema:"
                f"\n{output_schema}\n"
                "Your assessment should be as specific to the company as possible. "
                "For example, if the company is a subsidiary of some other company, "
                "match the industry based on the subsidiary's industry, not the parent company's. "
                "If none of the provided options are a good fit, label it as N/A."
                "The available industries are:\n\n{{industries}}.",
            ),
            ("human", "Here is the company profile: \n\n {{company_profile}}"),
        ],
        template_format="jinja2",
    )
    industry_mapper = industry_prompt | llm | JsonOutputParser()

    return industry_mapper.invoke(
        input={
            "industries": industries,
            "company_profile": company_profile.to_json(),
        }
    )

In [ ]:
results = []

In [ ]:
import json
from pathlib import Path
from joblib import Parallel, delayed


def process_single_company(idx, company_profile, results_dir):
    """Process one company and save result immediately"""
    result_file = Path(results_dir) / f"result_{idx}.json"

    # Skip if already processed
    if result_file.exists():
        print(f"Skipping {idx} (already processed)")
        return {"idx": idx, "status": "skipped"}

    try:
        response = get_census_industry(company_profile)

        # Save immediately
        with open(result_file, "w") as f:
            json.dump({"idx": idx, "result": response}, f, indent=2)

        return {"idx": idx, "status": "success", "result": response}

    except Exception as e:
        # Save error info
        error_file = Path(results_dir) / f"error_{idx}.json"
        with open(error_file, "w") as f:
            json.dump({"idx": idx, "error": str(e)}, f, indent=2)

        return {"idx": idx, "status": "failed", "error": str(e)}


def process_parallel_with_saves(company_profiles, results_dir="results"):
    """Process in parallel with individual saves"""
    Path(results_dir).mkdir(exist_ok=True)

    company_data = [(idx, row) for idx, row in company_profiles.iterrows()]

    # Process in parallel, each saving its own result
    results = Parallel(n_jobs=-1)(
        delayed(process_single_company)(idx, company_profile, results_dir)
        for idx, company_profile in tqdm(company_data, desc="Processing companies")
    )

    # Summarize results
    successful = [r for r in results if r["status"] == "success"]
    failed = [r for r in results if r["status"] == "failed"]
    skipped = [r for r in results if r["status"] == "skipped"]

    print(f"✅ Successful: {len(successful)}")
    print(f"❌ Failed: {len(failed)}")
    print(f"⏭️ Skipped: {len(skipped)}")

    return results


results = process_parallel_with_saves(company_profiles)

In [ ]:
results = pd.DataFrame.from_records(
    [json.load(file.open()) for file in sorted(Path("results").glob("result_*.json"))]
)
results = pd.json_normalize(results.result)
results

In [ ]:
def clean_dataframe(df):
    df_cleaned = df.copy()

    # Get all prefixed columns
    prefixed_cols = [col for col in df.columns if col.startswith("properties.")]

    for prefixed_col in prefixed_cols:
        base_col = prefixed_col.replace("properties.", "", 1)

        if base_col in df.columns:
            # Merge the columns
            df_cleaned[base_col] = df_cleaned[base_col].fillna(df_cleaned[prefixed_col])
            # Drop the prefixed column
            df_cleaned = df_cleaned.drop(columns=[prefixed_col])

    return df_cleaned


# Usage
df_cleaned = clean_dataframe(results)

In [ ]:
df_cleaned = df_cleaned.dropna(subset="id")

After cleaning, a few companies still need mapping. Re-run in a loop until all have been mapped.

In [ ]:
def map_remaining_companies(already_mapped):
    import shutil

    try:
        shutil.rmtree("results/remaining")
    except FileNotFoundError:
        pass

    remaining_profiles = company_profiles[
        ~company_profiles["id"].isin(already_mapped["id"])
    ]
    results = process_parallel_with_saves(
        remaining_profiles, results_dir="results/remaining"
    )
    results = pd.DataFrame.from_records(
        [
            json.load(file.open())
            for file in sorted(Path("results/remaining/").glob("result_*.json"))
        ]
    )
    results = pd.json_normalize(results.result)
    remaining_df_cleaned = clean_dataframe(results)
    remaining_df_cleaned = remaining_df_cleaned.dropna(subset="id")
    df = pd.concat([already_mapped, remaining_df_cleaned])
    return df

In [ ]:
while df_cleaned.shape[0] < company_profiles.shape[0]:
    df_cleaned = map_remaining_companies(df_cleaned)

In [ ]:
df_cleaned = df_cleaned[["id", "name", "assessment", "industry"]]

In [ ]:
industry_gender = industry_gender.set_index("industry")["women_pct"].to_dict()

In [ ]:
df_cleaned.loc[:, "women_pct"] = df_cleaned.industry.map(industry_gender)

In [ ]:
df_cleaned.to_csv("../data/derived/company_gender_share.csv", index=False)
df_cleaned.to_excel("../data/derived/company_gender_share.xlsx", index=False)

We can get some sense of accuracy by checking the hand-labeled industries against the automatically-labeled ones:

In [ ]:
reference = (
    pd.read_excel(
        "../data/supplemental/indus_gender_forsimon_cleaned.xlsx", na_values=["."]
    )[["employer", "indus_Simon", "indus_gen_percentage"]]
    .dropna()
    .drop_duplicates()
)

In [ ]:
overlap = df_cleaned.merge(reference, how="inner", left_on="name", right_on="employer")

In [ ]:
overlap.to_excel(
    "../data/derived/company_industry_mapping_ai_vs_human.xlsx", index=False
)